# 擬似対訳データを往復翻訳の一致度でふるいにかける

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_machine_translation/pseudo_parallel_filter.ipynb)

NLP クックブック 27-13 ／ 機械翻訳

逆翻訳で作った擬似対訳には、誤訳や意味の抜けが混ざる。質の低い組を自動的に取り除きたい。

In [1]:
!pip install -q datasets sentencepiece sacremoses sacrebleu sentence-transformers

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def load(name, **kwargs):
    return AutoTokenizer.from_pretrained(name, **kwargs), AutoModelForSeq2SeqLM.from_pretrained(name).eval()

ja_en = load("Helsinki-NLP/opus-mt-ja-en")                                       # 逆翻訳用 (日 → 英)
en_ja = load("facebook/nllb-200-distilled-600M", src_lang="eng_Latn")            # 訳し戻し用 (英 → 日)

@torch.no_grad()
def translate(pair, sentences, batch_size=16, **kwargs):
    tokenizer, model = pair
    out = []
    for i in range(0, len(sentences), batch_size):
        enc = tokenizer(sentences[i:i + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=128)
        out += tokenizer.batch_decode(model.generate(**enc, num_beams=4, max_new_tokens=128, **kwargs), skip_special_tokens=True)
    return out

ds = load_dataset("Helsinki-NLP/opus-100", "en-ja", split="test")
mono_ja = [ex["translation"]["ja"] for ex in ds if 8 <= len(ex["translation"]["ja"]) <= 60][:200]
pseudo_en = translate(ja_en, mono_ja)            # 逆翻訳: 日 → 英
round_trip = translate(en_ja, pseudo_en, forced_bos_token_id=en_ja[0].convert_tokens_to_ids("jpn_Jpan"))     # もう一度訳し戻す: 英 → 日
print(mono_ja[0], "→", pseudo_en[0], "→", round_trip[0])

もしそうなったら → If that happens, → もしそうなったら


In [3]:
import pandas as pd
import sacrebleu
from sentence_transformers import SentenceTransformer

# 指標 1: 元の日本語と、往復翻訳して戻ってきた日本語の一致度 (文単位の chrF)
chrf = [sacrebleu.sentence_chrf(rt, [ja]).score for rt, ja in zip(round_trip, mono_ja)]

# 指標 2: 日本語と、逆翻訳した英語の、多言語埋め込みによる類似度 (翻訳し直す必要がない)
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
e_ja = embedder.encode(["query: " + s for s in mono_ja], normalize_embeddings=True)
e_en = embedder.encode(["query: " + s for s in pseudo_en], normalize_embeddings=True)
cos = (e_ja * e_en).sum(axis=1)

df = pd.DataFrame({"ja": mono_ja, "pseudo_en": pseudo_en, "round_trip": round_trip, "chrF": chrf, "cos": cos})
print(df[["chrF", "cos"]].describe().round(3).T)
print("2 つの指標の相関:", round(df["chrF"].corr(df["cos"], method="spearman"), 3))
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 250)
print("--- 往復翻訳の一致度が低い組 (捨てる候補)")
print(df.sort_values("chrF").head(8)[["ja", "pseudo_en", "chrF", "cos"]].to_string(index=False))
print("--- 一致度が高い組")
print(df.sort_values("chrF").tail(5)[["ja", "pseudo_en", "chrF", "cos"]].to_string(index=False))

      count    mean     std    min    25%     50%     75%    max
chrF  200.0  25.539  20.693  0.000  9.239  19.671  37.611  100.0
cos   200.0   0.853   0.036  0.754  0.831   0.852   0.870    1.0
2 つの指標の相関: 0.244
--- 往復翻訳の一致度が低い組 (捨てる候補)
                                        ja                                                                                                                                                       pseudo_en     chrF      cos
                               フリスビー　知ってる？ Hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey. 0.000000 0.759395
                           今からまばたきしちゃダメだから                                                                                                                                          Don't blink right now. 0.000000 0.814826
                                  並 おかわり ！                                                          

In [4]:
# ふるいの効果を確かめる: 人手の英訳に対する逆翻訳の質 (chrF) が、残した組で上がっているか
refs = [ex["translation"]["en"] for ex in ds if 8 <= len(ex["translation"]["ja"]) <= 60][:200]
df["quality"] = [sacrebleu.sentence_chrf(h, [r]).score for h, r in zip(pseudo_en, refs)]
for metric in ["chrF", "cos"]:
    ranked = df.sort_values(metric, ascending=False)
    print(metric, {f"上位 {int(r * 100)}%": round(float(ranked.head(int(len(df) * r))["quality"].mean()), 1) for r in [0.25, 0.5, 0.75, 1.0]})

# 埋め込みの類似度が高すぎる組は、訳されずに原文がそのまま残っている (コピー) ことが多い
print(df[df["cos"] > 0.97][["ja", "pseudo_en", "cos"]].head(5).to_string(index=False))

chrF {'上位 25%': 37.7, '上位 50%': 35.3, '上位 75%': 35.2, '上位 100%': 34.8}
cos {'上位 25%': 44.4, '上位 50%': 41.0, '上位 75%': 38.5, '上位 100%': 34.8}
                                        ja                                  pseudo_en  cos
♪ But if you wanna leave, take good care ♪ ♪ But if you wanna leave, take good care ♪  1.0
         ♪ Yes, peace train, holy roller ♪          ♪ Yes, peace train, holy roller ♪  1.0
               ♪ to someone else's touch ♪                ♪ to someone else's touch ♪  1.0
                              ♪ Watch out!                               ♪ Watch out!  1.0


- 質の高い擬似対訳なら、訳し戻したときに元の文に近いものが返ってくるはず、という考え方。一致度の低い組は、逆翻訳の時点で意味が壊れている可能性が高い。
- 一致度の上位だけを学習に使う。しきい値を高くすると質は上がるが量が減るので、検証データの翻訳精度で決める。
- 往復翻訳は、2 つの翻訳モデルを通すので時間がかかる。多言語の文埋め込みの類似度なら 1 回の計算で済み、対訳コーパスからの雑音の除去にも同じように使える。この実験では、埋め込みの類似度で選んだほうが、残した組の質が高かった。
- 埋め込みの類似度は、訳されずに原文がそのまま残った組（コピー）に最高点を付けてしまう。類似度が高すぎる組や、原文と訳文の文字列がほぼ同じ組は、別に取り除く。
- 往復で元に戻るからといって、中間の訳が正しいとは限らない（同じ誤りを往復で打ち消すことがある）。逆に、正しい訳でも言い換えで戻ってくると一致度は下がる。あくまで目安として使う。
- 短い文や定型文ばかりが高得点で残りやすい。残したデータの長さや内容の分布が偏っていないかを確かめる。
- 長さ比・日本語文字率・数値一致に加え、双方向 IBM Model 1 アライメント信頼度スコアを用いた対訳ノイズフィルタリングと単語アライメント改善の演習は、姉妹編『[自然言語処理の教科書](https://github.com/mamorlis/nlpbook)』第 4 章の [統計的機械翻訳の基礎（IBM Model 1・フレーズベース SMT）・BLEU 評価・翻訳メモリ・対訳フィルタリング](https://github.com/mamorlis/nlpbook/blob/main/notebooks/ch4_smt_bleu.ipynb)（第 6 節）も参照。